In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"


In [ ]:

print("""
## log_error の定義

market_log_error_win = log(p_market_clipped / p_pred_clipped)

where:
  p_market_clipped = clip(p_market, 0.01, 0.99)
  p_pred_clipped = clip(p_pred, 0.01, 0.99)

クリップの目的:
  - p_pred = 0.001 の場合: log(p_actual/0.001) = 発散 → clip で 0.01 に制限
  - p_market = 0.999 の場合: log(0.999/0.1) の歪み → clip で 0.99 に制限
  → log_error ∈ [-4.6, 4.6] に収まる
""")


In [ ]:
print("""
## log_error の分布特性

期待される性質:
  - 平均 ≈ 0 (モデルが unbiased なら)
  - 標準偏差 ≈ 0.3-0.5
  - 正規分布に近い (QQプロットで確認)

正規性が重要な理由:
  - Stage2 の LightGBM は線形モデルの性質を持つ
  - 入力が正規分布に近いと学習が安定
  - 外れ値が少ないと汎化性能が向上
""")

In [ ]:
print("""
## Stage2 での寄与度

Stage2 の入力特徴量 (MarketModel.get_stage2_features()):
  1. signed_log_error_win — log_error の符号付き版
  2. abs_log_error_win — log_error の絶対値版
  3. market_error_rank_in_race — レース内での順位

SHAP summary plot で寄与度を確認:
  - log_error 系特徴量が top features に含まれること
  - 正規化が有効なら SHAP 値 > 0
""")

In [ ]:
print("""
## 結論: log_error 正規化

正規化の有効性:
  1. 両側クリップで発散防止
  2. 正規分布に近い入力で Stage2 の学習が安定
  3. SHAP で正の寄与度を確認

安全性:
  - クリップ範囲 [0.01, 0.99] で log_error ∈ [-4.6, 4.6]
  - これ以上の範囲の確率は極めて低い
""")